In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install langchain-chroma

In [ ]:
!pip install -q ragas==0.4.3 langchain==0.3.30 langchain-community==0.3.31 langchain-openai==0.3.35 rapidfuzz python-dotenv sentence-transformers pandas langchain-text-splitters
print("[✓] Tüm kütüphaneler başarıyla kuruldu.")

In [ ]:
!cp /kaggle/input/notebooks/""/generator-soru/outputs/chunk_map_20260829_060905.csv .
!cp /kaggle/input/notebooks/""/generator-soru/outputs/testset_mixed_102_soru_20260829_060905.csv .

In [ ]:
!mkdir outputs

In [ ]:
import os
import sys
import datetime
import time
import asyncio
import hashlib
import pandas as pd

from openai import OpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.messages import BaseMessage, AIMessage

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.run_config import RunConfig
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer, 
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

import shutil
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma

In [ ]:
print("="*60)
print("      CHROMA DB VEKTÖR VERİTABANI OLUŞTURULUYOR")
print("="*60)

# 1. Chunk haritasını (CSV) okuma
# NOT: Eğer CSV dosyanın adı farklıysa burayı güncellemeyi unutma!
chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"

In [ ]:
if not os.path.exists(chunk_map_path):
    print(f"🔴 HATA: '{chunk_map_path}' bulunamadı! Lütfen dosya yolunu kontrol et.")
else:
    print(f"[*] Chunk haritası okunuyor: {chunk_map_path}")
    df = pd.read_csv(chunk_map_path)
    
    # 2. DataFrame'i LangChain Document nesnelerine çevirme
    docs = []
    for _, row in df.iterrows():
        doc = Document(
            page_content=str(row['text']),
            metadata={
                "chunk_id": str(row['chunk_id']),
                "filename": str(row['filename']) 
            }
        )
        docs.append(doc)
        
    print(f"[✓] Toplam {len(docs)} parça başarıyla yüklendi. (ID'ler ve dosya adları korundu!)")

    # 3. ChromaDB için temizlik ve hazırlık
    db_dir = "/kaggle/working/chroma_db"

    # Hata almamak için eski klasör varsa siliyoruz
    if os.path.exists(db_dir):
        shutil.rmtree(db_dir)
        print("[*] Eski ChromaDB klasörü temizlendi.")

    try:
        import chromadb
        chromadb.api.client.SharedSystemClient.clear_system_cache()
    except Exception:
        pass

    # 4. BGE-M3 Embedding modelini yükleme
    model_name = "BAAI/bge-m3"
    print(f"[*] '{model_name}' modeli yükleniyor (Kaggle GPU kullanılacak)...")
    
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cuda'}, 
        encode_kwargs={'normalize_embeddings': True}
    )

    # 5. Veritabanını oluşturma ve kaydetme
    print(f"\n[*] Vektörler hesaplanıyor ve ChromaDB'ye yazılıyor...")
    print("[*] Bu işlem GPU hızına bağlı olarak 1-2 dakika sürebilir, lütfen bekle...")
    
    db = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=db_dir
    )

    print(f"\n[✓] Chroma veri tabanı başarıyla oluşturuldu!")
    print(f"[✓] Kayıt Yeri: {db_dir}")
    print("="*60)

    # 6. Kontrol için küçük bir test sorgusu
    query = "What is the diagnosis criteria for diabetes?" 
    print(f"\n🔎 TEST SORGUSU: '{query}'")
    
    results = db.similarity_search(query, k=3)

    for idx, doc in enumerate(results, 1):
        print(f"\n[{idx}] Eşleşme:")
        print(f"    - Chunk ID: {doc.metadata.get('chunk_id')}")
        print(f"    - Dosya Adı: {doc.metadata.get('filename')}")
        print(f"    - İçerik: {doc.page_content[:150]}...")

## test

In [ ]:
!cp /kaggle/input/notebooks/""/generator-soru/outputs/testset_mixed_102_soru_20260829_060905.csv .

In [ ]:
print("="*60)
print("      RECALL@5 ve PRECISION@5 HESAPLAMASI (CHUNK_ID İLE)")
print("="*60)

# 1. Chunk Map'i Yükleyip Text -> ID Sözlüğü Oluşturma
chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"
df_map = pd.read_csv(chunk_map_path)

# Metinleri ID'ye milimetrik eşleştirmek için bir sözlük (boşlukları temizliyoruz)
text_to_id = {}
for _, row in df_map.iterrows():
    clean_text = str(row['text']).strip()
    text_to_id[clean_text] = str(row['chunk_id'])

print(f"[*] {len(text_to_id)} parçalık Text-to-ID haritası oluşturuldu.")

# 2. ChromaDB'ye Bağlanma
db_dir = "/kaggle/working/chroma_db"
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'}, 
    encode_kwargs={'normalize_embeddings': True}
)
db = Chroma(persist_directory=db_dir, embedding_function=embeddings)
print("[✓] Vektör veritabanına başarıyla bağlanıldı.")

# 3. Test Setini Yükleme
testset_path = "/kaggle/working/testset_mixed_102_soru_20260829_060905.csv"
df_test = pd.read_csv(testset_path)

K = 5
recalls = []
precisions = []

print(f"[*] Toplam {len(df_test)} soru için CHUNK_ID eşleştirmesi ile hesaplanıyor...\n")

for index, row in df_test.iterrows():
    query = row['user_input']
    
    try:
        # Ragas'ın referans metinlerini al
        ground_truth_texts = ast.literal_eval(row['reference_contexts'])
    except Exception:
        continue
        
    # ADIM A: Beklenen (Ground Truth) Chunk ID'lerini bul
    expected_ids = []
    for gt_text in ground_truth_texts:
        clean_gt = str(gt_text).strip()
        if clean_gt in text_to_id:
            expected_ids.append(text_to_id[clean_gt])
            
    # Eğer beklenen ID bulunamadıysa (Ragas metni bozmuşsa vb.) atla
    if len(expected_ids) == 0:
        continue
        
    # ADIM B: Sistemden Arama Yap (K=5) ve Gelen ID'leri al
    retrieved_docs = db.similarity_search(query, k=K)
    retrieved_ids = [str(doc.metadata.get('chunk_id')) for doc in retrieved_docs]
    
    # ADIM C: ID KESİŞİMİ (İşte gerçek doğruluk testi!)
    true_positives = 0
    for exp_id in expected_ids:
        if exp_id in retrieved_ids:
            true_positives += 1
            
    # Metrikleri Hesaplama
    recall = true_positives / len(expected_ids)
    precision = true_positives / K
    
    recalls.append(recall)
    precisions.append(precision)

# 4. Sonuçların Ortalaması
avg_recall = np.mean(recalls)
avg_precision = np.mean(precisions)

print("-" * 60)
print(f"🎯 SİSTEMİN GERÇEK PERFORMANS KARNESİ (ID BAZLI - K={K})")
print("-" * 60)
print(f"Ortalama Recall@{K}    : {avg_recall:.4f}  ( %{avg_recall*100:.1f} )")
print(f"Ortalama Precision@{K} : {avg_precision:.4f}  ( %{avg_precision*100:.1f} )")
print("=" * 60)

In [ ]:
!pip install rank_bm25

## hibrit test

In [ ]:
import pandas as pd
import ast
import numpy as np
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

print("="*60)
print("      HİBRİT ARAMA İLE RECALL@5 & PRECISION@5 (ID BAZLI)")
print("="*60)

# 1. Chunk Map'i Yükleyip Text -> ID Sözlüğü ve BM25 Dokümanlarını Oluşturma
chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"
df_map = pd.read_csv(chunk_map_path)

text_to_id = {}
docs_for_bm25 = []

for _, row in df_map.iterrows():
    clean_text = str(row['text']).strip()
    chunk_id = str(row['chunk_id'])
    
    # Doğrulama (Ground Truth) haritası için
    text_to_id[clean_text] = chunk_id
    
    # BM25 (Kelime bazlı arama) motoru için
    docs_for_bm25.append(Document(
        page_content=clean_text,
        metadata={"chunk_id": chunk_id, "filename": str(row['filename'])}
    ))

print(f"[*] {len(text_to_id)} parçalık Text-to-ID haritası ve BM25 listesi hazır.")

# 2. Vektör (Chroma) Retriever Hazırlığı
db_dir = "/kaggle/working/chroma_db"
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'}, 
    encode_kwargs={'normalize_embeddings': True}
)
db = Chroma(persist_directory=db_dir, embedding_function=embeddings)
# Vektör motorundan 5 sonuç istiyoruz
chroma_retriever = db.as_retriever(search_kwargs={"k": 5})
print("[✓] Vektör (Semantik) motoru bağlandı.")

# 3. BM25 (Anahtar Kelime) Retriever Hazırlığı
bm25_retriever = BM25Retriever.from_documents(docs_for_bm25)
# Kelime motorundan da 5 sonuç istiyoruz
bm25_retriever.k = 5
print("[✓] BM25 (Anahtar Kelime) motoru eğitildi.")

# 4. HİBRİT ENSEMBLE RETRIEVER (Güçlerin Birleşimi)
# İki motorun sonuçlarını %50 - %50 ağırlıkla birleştiriyoruz (Reciprocal Rank Fusion)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.5, 0.5]
)
print("[✓] Hibrit Sistem Başarıyla Oluşturuldu!")

# 5. Test Seti Üzerinde Değerlendirme
testset_path = "/kaggle/working/testset_mixed_102_soru_20260829_060905.csv"
df_test = pd.read_csv(testset_path)

K = 5
recalls = []
precisions = []

print(f"\n[*] Toplam {len(df_test)} soru Hibrit Sistem ile test ediliyor (Kütfen bekleyin)...\n")

for index, row in df_test.iterrows():
    query = row['user_input']
    
    try:
        ground_truth_texts = ast.literal_eval(row['reference_contexts'])
    except Exception:
        continue
        
    expected_ids = []
    for gt_text in ground_truth_texts:
        clean_gt = str(gt_text).strip()
        if clean_gt in text_to_id:
            expected_ids.append(text_to_id[clean_gt])
            
    if len(expected_ids) == 0:
        continue
        
    # Hibrit motoru çalıştırıyoruz (Vektör 5 + BM25 5 parça getirip harmanlıyor)
    retrieved_docs = hybrid_retriever.invoke(query)
    
    # Kesin bir K=5 metriği ölçtüğümüz için en iyi ilk 5'i alıyoruz
    retrieved_docs = retrieved_docs[:K] 
    
    # Gelen dokümanların ID'lerini çıkarıyoruz
    retrieved_ids = [str(doc.metadata.get('chunk_id')) for doc in retrieved_docs]
    
    true_positives = 0
    for exp_id in expected_ids:
        if exp_id in retrieved_ids:
            true_positives += 1
            
    # Recall ve Precision Hesaplaması
    recall = true_positives / len(expected_ids)
    precision = true_positives / K
    
    recalls.append(recall)
    precisions.append(precision)

# 6. Sonuçların Ortalamasını Alma
avg_recall = np.mean(recalls)
avg_precision = np.mean(precisions)

print("-" * 60)
print(f"🔥 HİBRİT SİSTEMİN GERÇEK PERFORMANS KARNESİ (ID BAZLI - K={K})")
print("-" * 60)
print(f"Ortalama Recall@{K}    : {avg_recall:.4f}  ( %{avg_recall*100:.1f} )")
print(f"Ortalama Precision@{K} : {avg_precision:.4f}  ( %{avg_precision*100:.1f} )")
print("=" * 60)

In [ ]:
!!pip install sentence-transformers

# raranker

In [ ]:
import pandas as pd
import ast
import numpy as np
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# Reranker modülleri
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever

print("="*60)
print("  RERANKER DESTEKLİ HİBRİT ARAMA (RECALL & PRECISION)")
print("="*60)
print("="*60)
print("  RERANKER DESTEKLİ HİBRİT ARAMA (BGE-RERANKER-V2-M3)")
print("="*60)

# 1. Dosya Yolları Kontrolü (outputs klasörü kontrolü ile)
chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"
if not os.path.exists(chunk_map_path):
    chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"

testset_path = "/kaggle/working/testset_mixed_102_soru_20260829_060905.csv"
if not os.path.exists(testset_path):
    testset_path = "/kaggle/working/testset_mixed_102_soru_20260829_060905.csv"

# 2. Chunk Map ve Text-to-ID Hazırlığı
df_map = pd.read_csv(chunk_map_path)
text_to_id = {}
docs_for_bm25 = []

for _, row in df_map.iterrows():
    clean_text = str(row['text']).strip()
    chunk_id = str(row['chunk_id'])
    
    text_to_id[clean_text] = chunk_id
    docs_for_bm25.append(Document(
        page_content=clean_text,
        metadata={"chunk_id": chunk_id, "filename": str(row['filename'])}
    ))

print(f"[*] {len(text_to_id)} parçalık ID haritası hazır.")

# 3. Hibrit Taban (Kütüphaneci) Ayarları
INITIAL_K = 15
FINAL_K = 5

db_dir = "/kaggle/working/chroma_db"
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'}, 
    encode_kwargs={'normalize_embeddings': True}
)
db = Chroma(persist_directory=db_dir, embedding_function=embeddings)

chroma_retriever = db.as_retriever(search_kwargs={"k": INITIAL_K})
bm25_retriever = BM25Retriever.from_documents(docs_for_bm25)
bm25_retriever.k = INITIAL_K

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.5, 0.5]
)
print(f"[✓] İlk aşama Hibrit (K={INITIAL_K}) motoru kuruldu.")

# 4. Çok Dilli Reranker (GPU Destekli)
print("[*] Reranker modeli yükleniyor (BAAI/bge-reranker-v2-m3)...")
reranker_model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",
    model_kwargs={'device': 'cuda'}
)

compressor = CrossEncoderReranker(model=reranker_model, top_n=FINAL_K)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=hybrid_retriever
)
print(f"[✓] Çok dilli Reranker sistemi (Top {FINAL_K}) başarıyla aktif edildi!")

# 5. Test Seti Üzerinde Değerlendirme
df_test = pd.read_csv(testset_path)
recalls = []
precisions = []

print(f"\n[*] Toplam {len(df_test)} soru test ediliyor...\n")

for index, row in df_test.iterrows():
    query = row['user_input']
    
    try:
        ground_truth_texts = ast.literal_eval(row['reference_contexts'])
    except Exception:
        continue
        
    expected_ids = []
    for gt_text in ground_truth_texts:
        clean_gt = str(gt_text).strip()
        if clean_gt in text_to_id:
            expected_ids.append(text_to_id[clean_gt])
            
    if len(expected_ids) == 0:
        continue
        
    retrieved_docs = compression_retriever.invoke(query)
    retrieved_ids = [str(doc.metadata.get('chunk_id')) for doc in retrieved_docs]
    
    true_positives = 0
    for exp_id in expected_ids:
        if exp_id in retrieved_ids:
            true_positives += 1
            
    recall = true_positives / len(expected_ids)
    precision = true_positives / FINAL_K
    
    recalls.append(recall)
    precisions.append(precision)

avg_recall = np.mean(recalls)
avg_precision = np.mean(precisions)

print("-" * 60)
print(f"🌟 RERANKER V2-M3 PERFORMANS KARNESİ (ID BAZLI - K={FINAL_K})")
print("-" * 60)
print(f"Ortalama Recall@{FINAL_K}    : {avg_recall:.4f}  ( %{avg_recall*100:.1f} )")
print(f"Ortalama Precision@{FINAL_K} : {avg_precision:.4f}  ( %{avg_precision*100:.1f} )")
print("=" * 60)

# genel

In [ ]:
import ast
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar
from rapidfuzz import fuzz, process
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever

print("="*60)
print("   DERİNLEMESİNE RAG PERFORMANS VE İSTATİSTİK ANALİZİ (FUZZY-FIX)")
print("="*60)

# ------------------------------------------------------------
# 1. Dosya Yolları ve Veri Hazırlığı
# ------------------------------------------------------------
chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"
testset_path = "/kaggle/working/testset_mixed_102_soru_20260829_060905.csv"

df_map = pd.read_csv(chunk_map_path)
df_test = pd.read_csv(testset_path)

text_to_id = {}
docs_for_bm25 = []

for _, row in df_map.iterrows():
    clean_text = str(row['text']).strip()
    chunk_id = str(row['chunk_id'])
    text_to_id[clean_text] = chunk_id
    docs_for_bm25.append(Document(
        page_content=clean_text,
        metadata={"chunk_id": chunk_id, "filename": str(row['filename'])}
    ))

doc_texts_list = list(text_to_id.keys())

# ------------------------------------------------------------
# 2. FUZZY EŞLEŞTİRME FONKSİYONU (Exact -> Fuzzy Fallback)
# ------------------------------------------------------------
FUZZY_THRESHOLD = 90  # 85-95 arası ayarlanabilir

def match_text_to_id(gt_text, threshold=FUZZY_THRESHOLD):
    """Önce exact match dener, tutmazsa rapidfuzz ile en yakın chunk'ı bulur."""
    clean_gt = str(gt_text).strip()
    if clean_gt in text_to_id:
        return text_to_id[clean_gt], "exact", 100.0

    match = process.extractOne(clean_gt, doc_texts_list, scorer=fuzz.ratio)
    if match is not None:
        best_match, score, _ = match
        if score >= threshold:
            return text_to_id[best_match], "fuzzy", score
    return None, "no_match", 0.0

# ------------------------------------------------------------
# 3. DİAGNOSTİK: Eşleştirme öncesi ne kadar düşüyor, synthesizer bazında gör
# ------------------------------------------------------------
print("\n[*] Ön-diagnostik: context eşleştirme oranları hesaplanıyor...")
diag_rows = []
for _, row in df_test.iterrows():
    synth_name = row.get('synthesizer_name', 'Genel_Soru')
    try:
        ground_truth_texts = ast.literal_eval(row['reference_contexts'])
    except Exception:
        diag_rows.append({"synthesizer_name": synth_name, "status": "parse_fail"})
        continue

    match_types = [match_text_to_id(t)[1] for t in ground_truth_texts]
    if all(m == "no_match" for m in match_types):
        status = "zero_match"
    elif any(m == "fuzzy" for m in match_types):
        status = "has_fuzzy"
    else:
        status = "all_exact"
    diag_rows.append({"synthesizer_name": synth_name, "status": status})

diag_df = pd.DataFrame(diag_rows)
print("\n📋 Synthesizer bazında eşleştirme durumu:")
print(pd.crosstab(diag_df['synthesizer_name'], diag_df['status']))
print()

# ------------------------------------------------------------
# 4. Model Bağlantıları
# ------------------------------------------------------------
db_dir = "/kaggle/working/chroma_db"
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
db = Chroma(persist_directory=db_dir, embedding_function=embeddings)

K = 5
INITIAL_K_FOR_RERANK = 15

basic_retriever = db.as_retriever(search_kwargs={"k": K})

bm25_wide = BM25Retriever.from_documents(docs_for_bm25)
bm25_wide.k = INITIAL_K_FOR_RERANK
chroma_wide = db.as_retriever(search_kwargs={"k": INITIAL_K_FOR_RERANK})

hybrid_wide_retriever = EnsembleRetriever(
    retrievers=[bm25_wide, chroma_wide],
    weights=[0.5, 0.5]
)

reranker_model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",
    model_kwargs={'device': 'cuda'}
)
compressor = CrossEncoderReranker(model=reranker_model, top_n=K)
reranker_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_wide_retriever
)

# ------------------------------------------------------------
# 5. Test Döngüsü (Fuzzy Fallback ile)
# ------------------------------------------------------------
results_data = []
skip_count = 0

print(f"\n[*] Toplam {len(df_test)} soru analiz ediliyor...\n")

for index, row in df_test.iterrows():
    query = row['user_input']
    synth_name = row.get('synthesizer_name', row.get('question_type', 'Genel_Soru'))

    try:
        ground_truth_texts = ast.literal_eval(row['reference_contexts'])
    except Exception:
        skip_count += 1
        continue

    expected_ids = []
    match_info = []
    for gt_text in ground_truth_texts:
        matched_id, match_type, score = match_text_to_id(gt_text)
        if matched_id:
            expected_ids.append(matched_id)
            match_info.append((match_type, score))

    if len(expected_ids) == 0:
        skip_count += 1
        continue

    # Temel Vektör Testi
    res_basic = basic_retriever.invoke(query)
    ids_basic = [str(doc.metadata.get('chunk_id')) for doc in res_basic[:K]]
    tp_basic = sum(1 for exp_id in expected_ids if exp_id in ids_basic)
    recall_basic = tp_basic / len(expected_ids)

    # Hibrit Testi (ilk aşama, reranker'sız, K=5 kesilerek)
    res_hybrid = hybrid_wide_retriever.invoke(query)
    ids_hybrid = [str(doc.metadata.get('chunk_id')) for doc in res_hybrid[:K]]
    tp_hybrid = sum(1 for exp_id in expected_ids if exp_id in ids_hybrid)
    recall_hybrid = tp_hybrid / len(expected_ids)

    # Reranker Testi
    res_rerank = reranker_retriever.invoke(query)
    ids_rerank = [str(doc.metadata.get('chunk_id')) for doc in res_rerank[:K]]
    tp_rerank = sum(1 for exp_id in expected_ids if exp_id in ids_rerank)
    recall_rerank = tp_rerank / len(expected_ids)

    results_data.append({
        'query': query,
        'synthesizer_name': synth_name,
        'n_expected': len(expected_ids),
        'has_fuzzy_match': any(m[0] == "fuzzy" for m in match_info),
        'recall_basic': recall_basic,
        'recall_hybrid': recall_hybrid,
        'recall_rerank': recall_rerank
    })

df_res = pd.DataFrame(results_data)

print(f"[✓] Analiz tamamlanan soru sayısı : {len(df_res)}")
print(f"[!] Atlanan (eşleşme/parse hatası) : {skip_count}")
print(f"[*] Fuzzy eşleşme gerektiren soru sayısı: {df_res['has_fuzzy_match'].sum()}")

# ------------------------------------------------------------
# 6. WIN / LOSS / TIE ANALİZİ (Reranker vs Temel Vektör)
# ------------------------------------------------------------
wins = sum(df_res['recall_rerank'] > df_res['recall_basic'])
losses = sum(df_res['recall_rerank'] < df_res['recall_basic'])
ties = sum(df_res['recall_rerank'] == df_res['recall_basic'])

print("\n" + "-"*60)
print("📊 WIN / LOSS / TIE ANALİZİ (Reranker vs Temel Vektör)")
print("-"*60)
print(f"Reranker Kazandı (Win)  : {wins} soru")
print(f"Reranker Kaybetti (Loss): {losses} soru")
print(f"Berabere (Tie)          : {ties} soru")

# ------------------------------------------------------------
# 7. İSTATİSTİKSEL TESTLER
# ------------------------------------------------------------
stat, p_val = wilcoxon(df_res['recall_basic'], df_res['recall_rerank'])

df_res['bin_basic'] = (df_res['recall_basic'] == 1.0).astype(int)
df_res['bin_rerank'] = (df_res['recall_rerank'] == 1.0).astype(int)
contingency = pd.crosstab(df_res['bin_basic'], df_res['bin_rerank'])

try:
    mc_res = mcnemar(contingency, exact=True)
    mc_pvalue = mc_res.pvalue
except Exception:
    mc_pvalue = 1.0

print(f"\nWilcoxon p-değeri (Basic vs Rerank)   : {p_val:.5f}")
print(f"McNemar p-değeri (Basic vs Rerank)    : {mc_pvalue:.5f}")

stat_h, p_val_h = wilcoxon(df_res['recall_basic'], df_res['recall_hybrid'])
print(f"Wilcoxon p-değeri (Basic vs Hybrid)   : {p_val_h:.5f}")

# ------------------------------------------------------------
# 8. GENEL ORTALAMALAR
# ------------------------------------------------------------
print("\n" + "-"*60)
print("🎯 ORTALAMA RECALL@5 (Tüm Sistemler)")
print("-"*60)
print(f"Temel Vektör    : {df_res['recall_basic'].mean():.4f} ( %{df_res['recall_basic'].mean()*100:.1f} )")
print(f"Hibrit (ilk K)  : {df_res['recall_hybrid'].mean():.4f} ( %{df_res['recall_hybrid'].mean()*100:.1f} )")
print(f"Reranker'lı     : {df_res['recall_rerank'].mean():.4f} ( %{df_res['recall_rerank'].mean()*100:.1f} )")

# ------------------------------------------------------------
# 9. ALT GRUP (SYNTHESIZER) KIRILIMI
# ------------------------------------------------------------
print("\n" + "-"*60)
print("🔍 SORU TİPİNE (SYNTHESIZER) GÖRE ORTALAMA RECALL")
print("-"*60)
group_df = df_res.groupby('synthesizer_name')[['recall_basic', 'recall_hybrid', 'recall_rerank']].mean()
group_df['n_soru'] = df_res.groupby('synthesizer_name').size()
print(group_df.round(4))

# ------------------------------------------------------------
# 10. OUTLIER ANALİZİ
# ------------------------------------------------------------
print("\n" + "-"*60)
print("⚠️ OUTLIER ANALİZİ: Reranker'ın Temel Sistemden Kötü Kaldığı Sorular")
print("-"*60)
worse_questions = df_res[df_res['recall_rerank'] < df_res['recall_basic']]
print(f"Reranker'ın geriye götürdüğü toplam soru sayısı: {len(worse_questions)}")

if len(worse_questions) > 0:
    for idx, row in worse_questions.head(5).iterrows():
        print(f"\n- Soru: {row['query']}")
        print(f"  Temel Recall: {row['recall_basic']:.2f} | Reranker Recall: {row['recall_rerank']:.2f}")

# ------------------------------------------------------------
# 11. Sonuçları kaydet (ileride tekrar analiz için)
# ------------------------------------------------------------
df_res.to_csv("/kaggle/working/eval_results_detailed.csv", index=False, encoding="utf-8")
print("\n[✓] Detaylı sonuçlar kaydedildi: /kaggle/working/eval_results_detailed.csv")

print("="*60)

## Contexual rag db

In [ ]:
import os
import time
import asyncio
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import BaseMessage, AIMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from openai import OpenAI

print("="*60)
print("      ÖZEL AZURE RESPONSES İLE CONTEXTUAL RAG")
print("="*60)

# 1. Senin Özel LangChain Model Sınıfın
class CustomAzureResponsesModel(BaseChatModel):
    api_key: str = ""
    base_url: str = ""
    model_name: str = "gpt-4.1-mini"
    delay: float = 0.5 # İstekler arası küçük bir bekleme

    def _generate(self, messages: list[BaseMessage], stop: list[str] | None = None, **kwargs) -> ChatResult:
        time.sleep(self.delay)
        prompt_text = "\n".join([msg.content for msg in messages])
        
        client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        response = client.responses.create(model=self.model_name, input=prompt_text)
        
        output_text = getattr(response, "output_text", str(response))
        message = AIMessage(content=output_text)
        return ChatResult(generations=[ChatGeneration(message=message)])

    async def _agenerate(self, messages: list[BaseMessage], stop: list[str] | None = None, **kwargs) -> ChatResult:
        await asyncio.sleep(self.delay)
        prompt_text = "\n".join([msg.content for msg in messages])
        
        client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        response = client.responses.create(model=self.model_name, input=prompt_text)
        
        output_text = getattr(response, "output_text", str(response))
        message = AIMessage(content=output_text)
        return ChatResult(generations=[ChatGeneration(message=message)])

    @property
    def _llm_type(self) -> str:
        return "custom-azure-responses-openai"

# 2. Ham chunk haritasını yüklüyoruz
chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"
if not os.path.exists(chunk_map_path):
    chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"

df_map = pd.read_csv(chunk_map_path)
print(f"[*] Toplam {len(df_map)} parça özel Azure modeli ile zenginleştirilmek üzere yüklendi.")

# 3. Model ve Embedding Tanımlamaları
llm = CustomAzureResponsesModel()

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'}, 
    encode_kwargs={'normalize_embeddings': True}
)

# 4. İngilizce Orijinal Contextual RAG Prompt Şablonu
context_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert assistant analyzing academic and medical texts. Considering the source document name/context and the chunk provided, write a short, concise, 1-2 sentence contextual summary explaining what this chunk means within the document and where it fits. Provide only the context, no extra explanation."),
    ("user", "SOURCE FILE: {filename}\n\nCHUNK:\n{chunk_text}\n\nShort context summary for this chunk:")
])

contextual_chain = context_prompt | llm | StrOutputParser()

print("[*] Özel Azure LLM ile her parça için bağlam üretimi başlatılıyor...")

contextualized_docs = []

# 5. Tüm Parçaları Özel Model ile Zenginleştirme Döngüsü
for index, row in df_map.iterrows():
    filename = str(row['filename'])
    chunk_text = str(row['text'])
    chunk_id = str(row['chunk_id'])
    
    try:
        generated_context = contextual_chain.invoke({
            "filename": filename, 
            "chunk_text": chunk_text
        })
        
        contextualized_content = f"Context: {generated_context}\n\nContent: {chunk_text}"
    except Exception as e:
        print(f"[!] Hata oluştu (Satır {index}): {e}")
        contextualized_content = chunk_text
        
    contextualized_docs.append(Document(
        page_content=contextualized_content,
        metadata={"chunk_id": chunk_id, "filename": filename}
    ))

print(f"[✓] Toplam {len(contextualized_docs)} parça bağlamsal hale getirildi.")

# 6. Yeni Contextual ChromaDB Oluşturma ve Kaydetme
contextual_db_dir = "/kaggle/working/chroma_db_contextual"
print(f"[*] Yeni Contextual ChromaDB kaydediliyor: {contextual_db_dir}")

vector_db = Chroma.from_documents(
    documents=contextualized_docs,
    embedding=embeddings,
    persist_directory=contextual_db_dir
)

print("="*60)
print("[✓] ÖZEL MODEL DESTEKLİ CONTEXTUAL RAG VERİTABANI HAZIR!")
print("="*60)

### test contexual 

In [ ]:
import ast
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar
from rapidfuzz import fuzz, process
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever

print("="*60)
print("   4'LÜ RAG PERFORMANS VE İSTATİSTİK ANALİZİ (CONTEXTUAL DAHİL)")
print("="*60)

# ------------------------------------------------------------
# 1. Dosya Yolları ve Veri Hazırlığı
# ------------------------------------------------------------
chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"
testset_path = "/kaggle/working/testset_mixed_102_soru_20260829_060905.csv"

df_map = pd.read_csv(chunk_map_path)
df_test = pd.read_csv(testset_path)

text_to_id = {}
docs_for_bm25 = []

for _, row in df_map.iterrows():
    clean_text = str(row['text']).strip()
    chunk_id = str(row['chunk_id'])
    text_to_id[clean_text] = chunk_id
    docs_for_bm25.append(Document(
        page_content=clean_text,
        metadata={"chunk_id": chunk_id, "filename": str(row['filename'])}
    ))

doc_texts_list = list(text_to_id.keys())

# ------------------------------------------------------------
# 2. FUZZY EŞLEŞTİRME FONKSİYONU
# ------------------------------------------------------------
FUZZY_THRESHOLD = 90

def match_text_to_id(gt_text, threshold=FUZZY_THRESHOLD):
    clean_gt = str(gt_text).strip()
    if clean_gt in text_to_id:
        return text_to_id[clean_gt], "exact", 100.0

    match = process.extractOne(clean_gt, doc_texts_list, scorer=fuzz.ratio)
    if match is not None:
        best_match, score, _ = match
        if score >= threshold:
            return text_to_id[best_match], "fuzzy", score
    return None, "no_match", 0.0

# ------------------------------------------------------------
# 3. Model Bağlantıları ve Veritabanları
# ------------------------------------------------------------
db_dir = "/kaggle/working/chroma_db"
contextual_db_dir = "/kaggle/working/chroma_db_contextual" # Azure contextual DB yolu

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

db = Chroma(persist_directory=db_dir, embedding_function=embeddings)
db_contextual = Chroma(persist_directory=contextual_db_dir, embedding_function=embeddings)

K = 5
INITIAL_K_FOR_RERANK = 15

# Retriever Tanımları
basic_retriever = db.as_retriever(search_kwargs={"k": K})
contextual_retriever = db_contextual.as_retriever(search_kwargs={"k": K})

bm25_wide = BM25Retriever.from_documents(docs_for_bm25)
bm25_wide.k = INITIAL_K_FOR_RERANK
chroma_wide = db.as_retriever(search_kwargs={"k": INITIAL_K_FOR_RERANK})

hybrid_wide_retriever = EnsembleRetriever(
    retrievers=[bm25_wide, chroma_wide],
    weights=[0.5, 0.5]
)

# BM25 K=5 için (Hibrit testi için)
bm25_standard = BM25Retriever.from_documents(docs_for_bm25)
bm25_standard.k = K
chroma_standard = db.as_retriever(search_kwargs={"k": K})
hybrid_standard_retriever = EnsembleRetriever(
    retrievers=[bm25_standard, chroma_standard],
    weights=[0.5, 0.5]
)

print("[*] Reranker modeli yükleniyor...")
reranker_model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",
    model_kwargs={'device': 'cuda'}
)
compressor = CrossEncoderReranker(model=reranker_model, top_n=K)
reranker_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_wide_retriever
)

# ------------------------------------------------------------
# 4. Test Döngüsü (4 Sistem Dahil)
# ------------------------------------------------------------
results_data = []
skip_count = 0

print(f"\n[*] Toplam {len(df_test)} soru 4 sistem üzerinde test ediliyor...\n")

for index, row in df_test.iterrows():
    query = row['user_input']
    synth_name = row.get('synthesizer_name', row.get('question_type', 'Genel_Soru'))

    try:
        ground_truth_texts = ast.literal_eval(row['reference_contexts'])
    except Exception:
        skip_count += 1
        continue

    expected_ids = []
    match_info = []
    for gt_text in ground_truth_texts:
        matched_id, match_type, score = match_text_to_id(gt_text)
        if matched_id:
            expected_ids.append(matched_id)
            match_info.append((match_type, score))

    if len(expected_ids) == 0:
        skip_count += 1
        continue

    # 1. Temel Vektör Testi
    res_basic = basic_retriever.invoke(query)
    ids_basic = [str(doc.metadata.get('chunk_id')) for doc in res_basic[:K]]
    recall_basic = sum(1 for exp_id in expected_ids if exp_id in ids_basic) / len(expected_ids)

    # 2. Hibrit Testi
    res_hybrid = hybrid_standard_retriever.invoke(query)
    ids_hybrid = [str(doc.metadata.get('chunk_id')) for doc in res_hybrid[:K]]
    recall_hybrid = sum(1 for exp_id in expected_ids if exp_id in ids_hybrid) / len(expected_ids)

    # 3. Reranker Testi
    res_rerank = reranker_retriever.invoke(query)
    ids_rerank = [str(doc.metadata.get('chunk_id')) for doc in res_rerank[:K]]
    recall_rerank = sum(1 for exp_id in expected_ids if exp_id in ids_rerank) / len(expected_ids)

    # 4. Contextual RAG Testi
    res_ctx = contextual_retriever.invoke(query)
    ids_ctx = [str(doc.metadata.get('chunk_id')) for doc in res_ctx[:K]]
    recall_ctx = sum(1 for exp_id in expected_ids if exp_id in ids_ctx) / len(expected_ids)

    results_data.append({
        'query': query,
        'synthesizer_name': synth_name,
        'n_expected': len(expected_ids),
        'has_fuzzy_match': any(m[0] == "fuzzy" for m in match_info),
        'recall_basic': recall_basic,
        'recall_hybrid': recall_hybrid,
        'recall_rerank': recall_rerank,
        'recall_ctx': recall_ctx
    })

df_res = pd.DataFrame(results_data)

print(f"[✓] Analiz tamamlanan soru sayısı : {len(df_res)}")
print(f"[!] Atlanan (eşleşme/parse hatası) : {skip_count}")

# ------------------------------------------------------------
# 5. GENEL ORTALAMALAR (4'LÜ KARNE)
# ------------------------------------------------------------
print("\n" + "-"*60)
print("🎯 ORTALAMA RECALL@5 (4'Lü Sistem Karnesi)")
print("-"*60)
print(f"Temel Vektör      : {df_res['recall_basic'].mean():.4f} ( %{df_res['recall_basic'].mean()*100:.1f} )")
print(f"Hibrit (BM25+Vec) : {df_res['recall_hybrid'].mean():.4f} ( %{df_res['recall_hybrid'].mean()*100:.1f} )")
print(f"Reranker'lı       : {df_res['recall_rerank'].mean():.4f} ( %{df_res['recall_rerank'].mean()*100:.1f} )")
print(f"Contextual RAG    : {df_res['recall_ctx'].mean():.4f} ( %{df_res['recall_ctx'].mean()*100:.1f} )")

# ------------------------------------------------------------
# 6. İSTATİSTİKSEL TESTLER (Wilcoxon)
# ------------------------------------------------------------
_, p_br = wilcoxon(df_res['recall_basic'], df_res['recall_rerank'])
_, p_bc = wilcoxon(df_res['recall_basic'], df_res['recall_ctx'])
_, p_bh = wilcoxon(df_res['recall_basic'], df_res['recall_hybrid'])

print("\n" + "-"*60)
print("📊 İSTATİSTİKSEL ANLAMLILIK (Wilcoxon p-değerleri)")
print("-"*60)
print(f"Temel vs Reranker    p-değeri : {p_br:.5f}")
print(f"Temel vs Contextual  p-değeri : {p_bc:.5f}")
print(f"Temel vs Hibrit      p-değeri : {p_bh:.5f}")

# ------------------------------------------------------------
# 7. ALT GRUP (SYNTHESIZER) KIRILIMI
# ------------------------------------------------------------
print("\n" + "-"*60)
print("🔍 SORU TİPİNE GÖRE ORTALAMA RECALL KARŞILAŞTIRMASI")
print("-"*60)
group_df = df_res.groupby('synthesizer_name')[['recall_basic', 'recall_hybrid', 'recall_rerank', 'recall_ctx']].mean()
group_df['n_soru'] = df_res.groupby('synthesizer_name').size()
print(group_df.round(4))

# Sonuçları kaydet
df_res.to_csv("/kaggle/working/4_systems_eval_detailed.csv", index=False, encoding="utf-8")
print("\n[✓] 4 sistemin detaylı sonuçları kaydedildi: /kaggle/working/4_systems_eval_detailed.csv")
print("="*60)